# 🩻 How Gemini *Really* Selects and Calls a Tool — Raw HTTP, No SDK, Nothing Hidden

### Dinesh AI Academy | Day 4 — Agents & MCP (Zero-Abstraction Deep Dive)

**Why this notebook is different from [2-How_Gemini_Selects_And_Calls_Tools.ipynb](./2-How_Gemini_Selects_And_Calls_Tools.ipynb):**
that notebook used the official `google-genai` Python SDK. The SDK is real and fine to use in
production, but it is still a *layer* — it builds JSON for you, parses JSON for you, and hands you
friendly Python objects like `response.text` and `part.function_call`. That friendliness is exactly
what makes it feel like "the SDK is deciding something."

**This notebook deletes that layer.** We talk to the literal HTTPS endpoint
(`generativelanguage.googleapis.com`) using nothing but Python's `requests` library. Every request
body you see is the *exact* JSON that leaves your machine. Every response you see is the *exact,
unedited* JSON Google's servers send back — printed in full, including fields nobody usually shows
you. Nothing is summarized, nothing is filtered, nothing is renamed to look nicer.

**Learning objective:** by the end, you will be able to point at a literal key in a literal JSON
dictionary and say "*this* field, right here, is what people mean when they say 'the model selected
a tool'" and "*this* field, right here, is what people mean when they say 'the model decided on its
final answer.'" There is no third thing hiding anywhere else.

## 0. Ground rule for this notebook

We will call **one single REST method** for everything: `generateContent`. That's it. There is no
separate "tool-calling API" and no separate "chat API" — function calling is just `generateContent`
being sent a `tools` field in the request body, and the model *choosing*, as part of ordinary text
generation, to emit a special JSON shape instead of prose. You're about to watch that happen with
your own eyes, at the byte level.

In [1]:
!pip -q install -U requests python-dotenv

## 1. Setup — API key, endpoint, nothing else

No `google-genai` import anywhere in this notebook. Just `requests` and `json`.

In [2]:
import requests
import json
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")
if not GAISTUDIO_API_KEY:
    raise ValueError("GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file.")

MODEL = "gemini-3.5-flash-lite"

# This is the ENTIRE surface area we will ever talk to in this notebook.
# One REST method. No SDK. No hidden client object.
GENERATE_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

def raw_call(payload: dict) -> dict:
    """The only function in this notebook that touches the network.

    Sends `payload` EXACTLY as given, as the JSON body of an HTTPS POST.
    Returns the response body EXACTLY as Google sent it back -- as a plain
    Python dict from json, with nothing removed, renamed, or reshaped.

    No retry logic on purpose. You are running this notebook interactively,
    cell by cell -- YOU are the retry loop. If Google's servers return a 429
    (rate limited), `raise_for_status()` below will raise immediately with the
    real error message; read it, wait as long as it says, and just re-run this
    cell yourself. A hidden auto-retry would hide that decision from you, which
    is exactly the kind of thing this notebook exists to NOT hide.
    """
    response = requests.post(
        GENERATE_URL,
        params={"key": GAISTUDIO_API_KEY},
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    print(f"HTTP {response.status_code} {response.reason}  <-- this is the literal HTTP status line")
    response.raise_for_status()
    return response.json()

print("Endpoint:", GENERATE_URL)
print("Ready. No SDK has been imported in this notebook.")

Endpoint: https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent
Ready. No SDK has been imported in this notebook.


## 2. A `generateContent` call with *no* tools — the baseline

Before touching function calling, look at the rawest possible shape: a plain text prompt in, a
plain text answer out. Everything below is the literal wire format. Memorize this shape — every
later section is a small mutation of it.

In [3]:
payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "Say the single word: OK"}]}
    ]
}

print("----- EXACT REQUEST BODY SENT -----")
print(json.dumps(payload, indent=2))

raw_response = raw_call(payload)

print("\n----- EXACT RESPONSE BODY RECEIVED (completely unfiltered) -----")
print(json.dumps(raw_response, indent=2))

----- EXACT REQUEST BODY SENT -----
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "Say the single word: OK"
        }
      ]
    }
  ]
}


HTTP 200 OK  <-- this is the literal HTTP status line

----- EXACT RESPONSE BODY RECEIVED (completely unfiltered) -----
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "OK",
            "thoughtSignature": "El4KXAERTTIPA/zIqY9WPW7KbIs3no7/gJKzQqRSonfiG5NYqKptgHqx0WPgc+bSOcKYBBTvHZmtquIX7MRhxrd2XNQHWjkwgucSne9fq8b24PGZIp2lN1yRVxhb1cy4"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 7,
    "candidatesTokenCount": 1,
    "totalTokenCount": 8,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 7
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "responseId": "5xqsaqHpHer6juMPstjZsAc"
}


Point at the response above and name every top-level thing you see:

- `candidates` — a list, because you can technically ask for more than one candidate answer.
- `candidates[0].content.role` — always `"model"` for a response turn.
- `candidates[0].content.parts` — a **list**. This is the single most important fact in this
  notebook: an answer is not "text OR a function call", it is *a list of parts*, and each part
  independently happens to be one or the other. Nothing forces there to be only one part.
- `candidates[0].finishReason` — why generation stopped (`STOP` = the model chose to stop, on its
  own, same as any LLM emitting an end-of-turn token).
- `usageMetadata` — real token counts. Proof this is a plain autoregressive generation call, billed
  and measured exactly like every other one, not some special "decision engine" mode.
- (You may also see a `thoughtSignature` field inside a part — that's Gemini's opaque internal
  reasoning-state token. We are not filtering it out even though most tutorials do, because this
  notebook's whole point is to show you the response *unfiltered*. You don't need to decode it or
  do anything with it — just don't be surprised it's there.)

## 3. The tools we'll expose — real Python, zero AI

Three ordinary functions. Nothing about them knows Gemini exists.

In [4]:
def get_weather(city: str) -> dict:
    """Fake weather lookup so the notebook doesn't depend on a real weather API."""
    WEATHER_DB = {
        "tokyo": {"temp_c": 26, "condition": "sunny"},
        "paris": {"temp_c": 18, "condition": "cloudy"},
        "mumbai": {"temp_c": 31, "condition": "humid, partly cloudy"},
    }
    data = WEATHER_DB.get(city.strip().lower(), {"temp_c": 20, "condition": "unknown"})
    return {"city": city, **data}

def celsius_to_fahrenheit(celsius: float) -> dict:
    return {"celsius": celsius, "fahrenheit": round(celsius * 9 / 5 + 32, 1)}

def get_current_time(timezone: str) -> dict:
    from datetime import datetime
    from zoneinfo import ZoneInfo
    now = datetime.now(ZoneInfo(timezone))
    return {"timezone": timezone, "current_time": now.strftime("%A, %d %B %Y, %I:%M:%S %p")}

# Plain Python. Try them. No network call, no Gemini involved.
print(get_weather("Tokyo"))
print(celsius_to_fahrenheit(26))
print(get_current_time("Asia/Tokyo"))

{'city': 'Tokyo', 'temp_c': 26, 'condition': 'sunny'}
{'celsius': 26, 'fahrenheit': 78.8}
{'timezone': 'Asia/Tokyo', 'current_time': 'Friday, 18 September 2026, 01:52:56 AM'}


## 4. Hand-writing the raw `tools` JSON — no introspection, no magic

We are not letting any library guess this from type hints. We type the exact JSON, by hand, so
there is zero ambiguity about what crosses the network. Compare this to Section 3 above: **notice
there is no function body anywhere in this JSON.** Google's servers will never see `WEATHER_DB`,
never see the `round(...)` call, never see `ZoneInfo`. Only the interface.

In [5]:
TOOLS = [
    {
        "functionDeclarations": [
            {
                "name": "get_weather",
                "description": "Gets the current temperature (Celsius) and condition for a city.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name, e.g. 'Tokyo'."}
                    },
                    "required": ["city"],
                },
            },
            {
                "name": "celsius_to_fahrenheit",
                "description": "Converts a Celsius temperature to Fahrenheit.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "celsius": {"type": "number", "description": "Temperature in Celsius."}
                    },
                    "required": ["celsius"],
                },
            },
            {
                "name": "get_current_time",
                "description": "Gets the current local date and time for an IANA timezone name.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "timezone": {"type": "string", "description": "IANA timezone, e.g. 'Asia/Tokyo'."}
                    },
                    "required": ["timezone"],
                },
            },
        ]
    }
]

# This -- and only this -- JSON is what tells Gemini these tools exist.
print(json.dumps(TOOLS, indent=2))

[
  {
    "functionDeclarations": [
      {
        "name": "get_weather",
        "description": "Gets the current temperature (Celsius) and condition for a city.",
        "parameters": {
          "type": "object",
          "properties": {
            "city": {
              "type": "string",
              "description": "City name, e.g. 'Tokyo'."
            }
          },
          "required": [
            "city"
          ]
        }
      },
      {
        "name": "celsius_to_fahrenheit",
        "description": "Converts a Celsius temperature to Fahrenheit.",
        "parameters": {
          "type": "object",
          "properties": {
            "celsius": {
              "type": "number",
              "description": "Temperature in Celsius."
            }
          },
          "required": [
            "celsius"
          ]
        }
      },
      {
        "name": "get_current_time",
        "description": "Gets the current local date and time for an IANA timezone name

## 5. Watching "tool selection" happen — one raw request, one raw response

Same `generateContent` method as Section 2. The **only** difference in the request is the added
`tools` field. Watch the response shape change accordingly.

In [6]:
payload = {
    "contents": [
        {"role": "user", "parts": [{"text": "What's the weather in Mumbai right now?"}]}
    ],
    "tools": TOOLS,
}

print("----- EXACT REQUEST BODY SENT -----")
print(json.dumps(payload, indent=2))

raw_response = raw_call(payload)

print("\n----- EXACT RESPONSE BODY RECEIVED (completely unfiltered) -----")
print(json.dumps(raw_response, indent=2))

----- EXACT REQUEST BODY SENT -----
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "What's the weather in Mumbai right now?"
        }
      ]
    }
  ],
  "tools": [
    {
      "functionDeclarations": [
        {
          "name": "get_weather",
          "description": "Gets the current temperature (Celsius) and condition for a city.",
          "parameters": {
            "type": "object",
            "properties": {
              "city": {
                "type": "string",
                "description": "City name, e.g. 'Tokyo'."
              }
            },
            "required": [
              "city"
            ]
          }
        },
        {
          "name": "celsius_to_fahrenheit",
          "description": "Converts a Celsius temperature to Fahrenheit.",
          "parameters": {
            "type": "object",
            "properties": {
              "celsius": {
                "type": "number",
                "description

HTTP 200 OK  <-- this is the literal HTTP status line

----- EXACT RESPONSE BODY RECEIVED (completely unfiltered) -----
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "functionCall": {
              "name": "get_weather",
              "args": {
                "city": "Mumbai"
              },
              "id": "call_41788"
            },
            "thoughtSignature": "El4KXAERTTIP8N9iSKOJCj7AEwQbefaRtG7Cjt9xl0Lsti5A4YKQeuJesI1Pr8Nh2/Q3YqSu7MutLwOuEETdZd77ZnCezONToGn3iC/tfwVtUegSauQvcJeSCRlJ0dzy"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0,
      "finishMessage": "Model generated function call(s)."
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 206,
    "candidatesTokenCount": 16,
    "totalTokenCount": 222,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 206
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3

### This is the naked truth of "tool selection"

Go find, inside the JSON you just printed, this exact path:

```text
candidates[0].content.parts[?].functionCall = {"name": "get_weather", "args": {"city": "Mumbai"}}
```

That dictionary is the **entire mechanism**. There is no other place, no other signal, no hidden
flag anywhere in the response that says "tool selected." It is one JSON key
(`functionCall`) appearing on one part instead of the `text` key appearing. Compare:

| | No tool needed (Section 2) | Tool selected (this section) |
|---|---|---|
| Part shape | `{"text": "..."}` | `{"functionCall": {"name": ..., "args": {...}}}` |
| `finishReason` | `STOP` | `STOP` (**same value** — a function call is a *completed*, ordinary generation, not an interruption) |
| What decided it | The prompt made "emit prose" the most likely continuation | The prompt + the `tools` schema made "emit this specific JSON structure" the most likely continuation |

There is no `if user_wants_weather:` anywhere on Google's servers. The model was trained so that,
given a conversation and a list of function schemas, producing a `functionCall`-shaped part is
sometimes literally the highest-probability next tokens to generate — exactly the same statistical
process as generating the word "OK" in Section 2. **"Selecting a tool" is not a different kind of
computation. It's the same computation, aimed at a different output shape.**

(You may also notice a `functionCall.id` field, e.g. `"call_41788"`, in the raw response above — a
correlation id the API attaches so that, when several tool calls happen in the same turn, you can
match each `functionResponse` back to the call it answers. It doesn't change anything explained
above; it's shown here, unfiltered, precisely because it's part of the real, live response and this
notebook doesn't hide fields just because they weren't in the plan.)

## 6. Extracting the request, by hand, with zero SDK helpers

No `part.function_call.name`. No `.args`. Just dictionary indexing, because that is genuinely all
`part.function_call` ever was under the hood.

In [7]:
parts = raw_response["candidates"][0]["content"]["parts"]
function_call_part = next(p for p in parts if "functionCall" in p)

tool_name = function_call_part["functionCall"]["name"]
tool_args = function_call_part["functionCall"]["args"]

print("Requested tool name (plain string):", tool_name)
print("Requested args (plain dict):       ", tool_args)
print("Type of tool_name:", type(tool_name), " Type of tool_args:", type(tool_args))

# At this exact line, Google's servers are 100% finished. Nothing further happens
# there for this turn. Everything from here on is OUR process, OUR machine.
print("\n>>> Google's job for this turn ended the moment the HTTP response above arrived. <<<")

Requested tool name (plain string): get_weather
Requested args (plain dict):        {'city': 'Mumbai'}
Type of tool_name: <class 'str'>  Type of tool_args: <class 'dict'>

>>> Google's job for this turn ended the moment the HTTP response above arrived. <<<


## 7. Executing the tool — a dictionary lookup, nothing more

This is deliberately the most boring code in the whole notebook. That's the point: nothing
"intelligent" happens here. It's 2015-era plugin-system code.

In [8]:
TOOLBOX = {
    "get_weather": get_weather,
    "celsius_to_fahrenheit": celsius_to_fahrenheit,
    "get_current_time": get_current_time,
}

def execute_tool(name: str, args: dict):
    func = TOOLBOX.get(name)
    if func is None:
        raise ValueError(
            f"Gemini's response named a tool called '{name}', but our TOOLBOX doesn't have it. "
            f"This is OUR error, raised on OUR machine, purely from reading a string out of JSON. "
            f"Google's servers do not know, and will never know, that this failed."
        )
    return func(**args)

tool_result = execute_tool(tool_name, tool_args)
print("Tool executed locally. Return value:", tool_result)

Tool executed locally. Return value: {'city': 'Mumbai', 'temp_c': 31, 'condition': 'humid, partly cloudy'}


### Proof: an unregistered tool name fails **locally**, not on Google's side

In [9]:
try:
    execute_tool("get_forecast", {"city": "Tokyo"})  # never declared, never registered
except ValueError as e:
    print("Caught locally, in our own Python process:")
    print(" ", e)

Caught locally, in our own Python process:
  Gemini's response named a tool called 'get_forecast', but our TOOLBOX doesn't have it. This is OUR error, raised on OUR machine, purely from reading a string out of JSON. Google's servers do not know, and will never know, that this failed.


## 8. Sending the tool result back — building turn 2's raw JSON by hand

To let Gemini use the result, we don't call some special "submit result" endpoint. We call
`generateContent` **again**, with a longer `contents` list: the original question, then the
model's own function-call turn appended back verbatim (role `"model"`), then a new turn holding a
`functionResponse` part (role `"user"`). The model has no memory between calls — the *entire*
conversation state is just this list, re-sent in full, every single time.

In [10]:
payload_turn_2 = {
    "contents": [
        # Turn 1: our original question
        {"role": "user", "parts": [{"text": "What's the weather in Mumbai right now?"}]},
        # Turn 2: Gemini's OWN prior response, played back to it verbatim.
        # We copy this straight out of raw_response -- we don't reconstruct it by hand,
        # because it must match exactly what the model said, including args formatting.
        raw_response["candidates"][0]["content"],
        # Turn 3: OUR reply -- the tool's result, wrapped as a functionResponse part.
        {
            "role": "user",
            "parts": [
                {
                    "functionResponse": {
                        "name": tool_name,
                        "response": tool_result,
                    }
                }
            ],
        },
    ],
    "tools": TOOLS,
}

print("----- EXACT REQUEST BODY SENT (note: 3 full turns, tool result included as data) -----")
print(json.dumps(payload_turn_2, indent=2))

----- EXACT REQUEST BODY SENT (note: 3 full turns, tool result included as data) -----
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "What's the weather in Mumbai right now?"
        }
      ]
    },
    {
      "parts": [
        {
          "functionCall": {
            "name": "get_weather",
            "args": {
              "city": "Mumbai"
            },
            "id": "call_41788"
          },
          "thoughtSignature": "El4KXAERTTIP8N9iSKOJCj7AEwQbefaRtG7Cjt9xl0Lsti5A4YKQeuJesI1Pr8Nh2/Q3YqSu7MutLwOuEETdZd77ZnCezONToGn3iC/tfwVtUegSauQvcJeSCRlJ0dzy"
        }
      ],
      "role": "model"
    },
    {
      "role": "user",
      "parts": [
        {
          "functionResponse": {
            "name": "get_weather",
            "response": {
              "city": "Mumbai",
              "temp_c": 31,
              "condition": "humid, partly cloudy"
            }
          }
        }
      ]
    }
  ],
  "tools": [
    {
      

In [11]:
raw_response_2 = raw_call(payload_turn_2)

print("\n----- EXACT RESPONSE BODY RECEIVED (completely unfiltered) -----")
print(json.dumps(raw_response_2, indent=2))

final_text = raw_response_2["candidates"][0]["content"]["parts"][0]["text"]
print("\nFinal answer text (candidates[0].content.parts[0].text):")
print(" ", final_text)

HTTP 200 OK  <-- this is the literal HTTP status line

----- EXACT RESPONSE BODY RECEIVED (completely unfiltered) -----
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "The current weather in Mumbai is 31\u00b0C (humid, partly cloudy).",
            "thoughtSignature": "El4KXAERTTIP9Bu2/PjPrhL7MGI8UHxSLj+JYZixDSHLuK5tonZvZmNeCdldIuTVui7qk/rYAdNi9LwCb1S9Tl0ZGmcUvOhYFdyH6uXmwZNFN/1+xbeLSCJyVE6yBVHW"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 252,
    "candidatesTokenCount": 17,
    "totalTokenCount": 269,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 252
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "responseId": "6hqsarr7L5WwjuMPwdPO4AE"
}

Final answer text (candidates[0].content.parts[0].text):
  The current weather in Mumbai is 31°C (

### This is the naked truth of "deciding the final answer"

Look at `payload_turn_2` again: it is just a longer `contents` array. The model was handed the same
kind of input as Section 2 — a list of conversation turns — except this time one of those turns
happens to contain a dict under `"functionResponse"` instead of under `"text"`.

There is **no separate "synthesize final answer" step, function, or mode.** The model runs the
exact same generation process as every other call in this notebook: read everything in `contents`
as context, predict the next tokens, stop when `finishReason` says `STOP`. This time, the
highest-probability continuation happens to be prose describing `{"temp_c": 31, "condition":
"humid, partly cloudy"}` in English, instead of another `functionCall` JSON structure. **"Deciding
the final answer" and "selecting a tool" are literally the same operation** — next-token
prediction over a growing `contents` list — producing two different, equally unremarkable JSON
shapes.

## 9. `toolConfig` — the one knob that overrides the model's free choice, in raw JSON

You can stop leaving "should I call a tool" purely to the model. `toolConfig.functionCallingConfig.mode`
has three literal string values Google's servers actually check:

| Raw JSON value | Behaviour |
|---|---|
| `"AUTO"` | Default — model chooses text vs. function call itself (everything above used this implicitly) |
| `"ANY"` | Model **must** emit a `functionCall` part this turn — optionally restricted via `allowedFunctionNames` |
| `"NONE"` | Model is **forbidden** from emitting a `functionCall` part this turn, even if one would help |

Watch `mode: "ANY"` force a tool call on a prompt that has nothing to do with weather:

In [12]:
payload_forced = {
    "contents": [
        {"role": "user", "parts": [{"text": "Tell me an interesting fact about octopuses."}]}
    ],
    "tools": TOOLS,
    "toolConfig": {
        "functionCallingConfig": {
            "mode": "ANY",
            "allowedFunctionNames": ["get_weather"],
        }
    },
}

print("----- EXACT REQUEST BODY SENT -----")
print(json.dumps(payload_forced, indent=2))

raw_forced = raw_call(payload_forced)
print("\n----- EXACT RESPONSE BODY RECEIVED -----")
print(json.dumps(raw_forced, indent=2))

----- EXACT REQUEST BODY SENT -----
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "Tell me an interesting fact about octopuses."
        }
      ]
    }
  ],
  "tools": [
    {
      "functionDeclarations": [
        {
          "name": "get_weather",
          "description": "Gets the current temperature (Celsius) and condition for a city.",
          "parameters": {
            "type": "object",
            "properties": {
              "city": {
                "type": "string",
                "description": "City name, e.g. 'Tokyo'."
              }
            },
            "required": [
              "city"
            ]
          }
        },
        {
          "name": "celsius_to_fahrenheit",
          "description": "Converts a Celsius temperature to Fahrenheit.",
          "parameters": {
            "type": "object",
            "properties": {
              "celsius": {
                "type": "number",
                "descri

HTTP 200 OK  <-- this is the literal HTTP status line

----- EXACT RESPONSE BODY RECEIVED -----
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "functionCall": {
              "name": "get_weather",
              "args": {
                "city": "Seattle"
              },
              "id": "call_3565"
            },
            "thoughtSignature": "El4KXAERTTIPHh+qHupuxfzKD9Cr7/ZqcXte25WTUyyAo2GMwzwgWhyq6W8zpC3mObKRuDAJjUR8pWMP7YZzVcKrYH8NL202osZi17riUVIjx4XSZbj+QTKsGVc3VR3E"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0,
      "finishMessage": "Model generated function call(s)."
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 206,
    "candidatesTokenCount": 16,
    "totalTokenCount": 222,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 206
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "respo

Notice the model had to invent *some* city for `args.city`, because `mode: "ANY"` removed its
only other option (plain text). That single string, `"ANY"`, sitting inside `toolConfig`, is the
entire mechanism — there's no separate "force mode" endpoint or flag anywhere else.

## 10. Parallel tool calls — one turn, multiple `functionCall` parts

Section 5 showed one `functionCall` part. Nothing in the API says there can only be one. If a
prompt needs two *independent* tool calls (neither needs the other's result to know its own
arguments), Gemini can put **two `functionCall` parts in the same `parts` list, in the same
response**, in a single round trip.

In [13]:
payload_parallel = {
    "contents": [
        {"role": "user", "parts": [{
            "text": "What's the weather in Paris, and separately, what time is it in Asia/Tokyo? "
                     "These two things are unrelated to each other."
        }]}
    ],
    "tools": TOOLS,
}

raw_parallel = raw_call(payload_parallel)
print(json.dumps(raw_parallel, indent=2))

call_parts = [p for p in raw_parallel["candidates"][0]["content"]["parts"] if "functionCall" in p]
print(f"\n{len(call_parts)} functionCall part(s) in this ONE response:")
for p in call_parts:
    print(" -", p["functionCall"]["name"], p["functionCall"]["args"])

HTTP 200 OK  <-- this is the literal HTTP status line
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "functionCall": {
              "name": "get_weather",
              "args": {
                "city": "Paris"
              },
              "id": "call_54481"
            },
            "thoughtSignature": "El4KXAERTTIPWwrBGlgxQCxRrzIdnacSOALfWifLD64NWA99Em9LCA0SKvPgnpvGdf8U12bzHP+8ubZQNCHRgYhqg2aFlG+q7Z7byY8pqQcf5jpi4tu7PQgmZiq29vUD"
          },
          {
            "functionCall": {
              "name": "get_current_time",
              "args": {
                "timezone": "Asia/Tokyo"
              },
              "id": "call_54482"
            }
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0,
      "finishMessage": "Model generated function call(s)."
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 225,
    "candidatesTokenCount": 36,
    "totalTokenCount": 261,
    "prom

If you got two parts above: that's the model predicting, in a single forward pass, that the most
likely continuation contains *two* `functionCall`-shaped structures back to back. Still no loop, no
special "batch mode" — just a longer `parts` list in the same response shape you've seen all
notebook.

To answer such a prompt fully, your own code must execute **both**, then send back **both**
`functionResponse` parts in the next turn's `contents` (a list of two dicts under one `"user"`
role turn, or two separate turns — both are valid; Section 11's loop below sends them as one
turn, matching how they arrived).

## 11. The full loop — hand-written, every request/response printed, nothing automatic

This is the same idea as the SDK's "automatic function calling," except every single row is code
*you* wrote and can see. No hidden iteration happens anywhere outside this `while` loop.

In [14]:
def run_agent_loop(user_text: str, max_iterations: int = 5, verbose: bool = True):
    contents = [{"role": "user", "parts": [{"text": user_text}]}]

    for iteration in range(1, max_iterations + 1):
        if verbose:
            print(f"\n================ ROUND TRIP {iteration}: SENDING ================")
            print(json.dumps({"contents": contents, "tools": TOOLS}, indent=2)[:2000], "...")

        response = raw_call({"contents": contents, "tools": TOOLS})

        if verbose:
            print(f"\n---------------- ROUND TRIP {iteration}: RAW RESPONSE ----------------")
            print(json.dumps(response, indent=2))

        model_turn = response["candidates"][0]["content"]
        contents.append(model_turn)  # play the model's own turn back into history, verbatim

        call_parts = [p for p in model_turn["parts"] if "functionCall" in p]

        if not call_parts:
            final_text = next(p["text"] for p in model_turn["parts"] if "text" in p)
            print(f"\n*** No functionCall part this round -> FINAL ANSWER reached at round trip {iteration} ***")
            return final_text

        print(f"\n{len(call_parts)} tool call(s) requested this round trip:")
        response_parts = []
        for part in call_parts:
            name = part["functionCall"]["name"]
            args = part["functionCall"]["args"]
            print(f"  -> executing {name}({args}) locally ...")
            result = execute_tool(name, args)
            print(f"     result: {result}")
            response_parts.append({"functionResponse": {"name": name, "response": result}})

        contents.append({"role": "user", "parts": response_parts})

    raise RuntimeError(f"Did not reach a final text answer within {max_iterations} round trips.")


answer = run_agent_loop(
    "What is today's weather in Tokyo, converted to Fahrenheit?"
)
print("\n================ FINAL ANSWER ================")
print(answer)


================ ROUND TRIP 1: SENDING ================
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "What is today's weather in Tokyo, converted to Fahrenheit?"
        }
      ]
    }
  ],
  "tools": [
    {
      "functionDeclarations": [
        {
          "name": "get_weather",
          "description": "Gets the current temperature (Celsius) and condition for a city.",
          "parameters": {
            "type": "object",
            "properties": {
              "city": {
                "type": "string",
                "description": "City name, e.g. 'Tokyo'."
              }
            },
            "required": [
              "city"
            ]
          }
        },
        {
          "name": "celsius_to_fahrenheit",
          "description": "Converts a Celsius temperature to Fahrenheit.",
          "parameters": {
            "type": "object",
            "properties": {
              "celsius": {
                "type"

HTTP 200 OK  <-- this is the literal HTTP status line

---------------- ROUND TRIP 1: RAW RESPONSE ----------------
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "functionCall": {
              "name": "get_weather",
              "args": {
                "city": "Tokyo"
              },
              "id": "call_37655"
            },
            "thoughtSignature": "El4KXAERTTIP686ENKS8IKFGhd2wK1tSfhcw1e2J47GQH7gIBho8pxYH513iGpLDMZTqwy0SGTvxg4k45obvNWso2lMbJrff7LrId4nwOMtAsB7P8Mzgl7dq5mkKw8qJ"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0,
      "finishMessage": "Model generated function call(s)."
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 209,
    "candidatesTokenCount": 16,
    "totalTokenCount": 225,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 209
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-fl

HTTP 200 OK  <-- this is the literal HTTP status line

---------------- ROUND TRIP 2: RAW RESPONSE ----------------
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "functionCall": {
              "name": "celsius_to_fahrenheit",
              "args": {
                "celsius": 26
              },
              "id": "call_39176"
            },
            "thoughtSignature": "El4KXAERTTIPDIEjG2Azv5ltjxr4jGjRRAnQzpefQoVKKwJLBwICEbrDLdnjmgyVosW8inoVZ6LJcv3heuirUnkHWT+AxuovPF408OVa3xzYUCJJca4XwGuQgBez9Da8"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0,
      "finishMessage": "Model generated function call(s)."
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 252,
    "candidatesTokenCount": 20,
    "totalTokenCount": 272,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 252
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemin

HTTP 200 OK  <-- this is the literal HTTP status line

---------------- ROUND TRIP 3: RAW RESPONSE ----------------
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "text": "Today's weather in Tokyo is sunny with a temperature of 26\u00b0C, which is 78.8\u00b0F.",
            "thoughtSignature": "El4KXAERTTIPGWB8pTwXkj41LKUSFv/prnyS45+GLios80NYnduupuofSxRQ4SRLvNzv4hG/JRK6rC3hQnIShAiQULj5nlMEx72gWMhq4EgRhnLNMFKYeA9PcjFRytAX"
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": {
    "promptTokenCount": 298,
    "candidatesTokenCount": 28,
    "totalTokenCount": 326,
    "promptTokensDetails": [
      {
        "modality": "TEXT",
        "tokenCount": 298
      }
    ],
    "serviceTier": "standard"
  },
  "modelVersion": "gemini-3.5-flash-lite",
  "responseId": "8hqsaoHJMdvdjuMPn_qMyAE"
}

*** No functionCall part this round -> FINAL ANSWER reached at round trip 3 ***


Read the printed round trips top to bottom. You should be able to point at exactly:

- **Round trip 1**: model requests `get_weather` (it needs the temperature before it can convert
  anything — this is *why* it can't ask for the conversion in the same round trip as Section 10's
  independent example; the conversion's argument doesn't exist yet).
- Our loop executes `get_weather` locally, appends a `functionResponse` turn.
- **Round trip 2**: model requests `celsius_to_fahrenheit`, now that it has a Celsius value to pass.
- Our loop executes it locally, appends another `functionResponse` turn.
- **Round trip 3**: model turn has zero `functionCall` parts, only `text` -> loop returns.

Every one of those round trips is one plain `generateContent` HTTP call with a longer `contents`
list than the last. There is no step anywhere that isn't either "HTTP request/response" or
"our own Python function running."

## 12. 🎤 Naked cheat sheet — literal JSON fields, no paraphrasing

| Concept people describe with a mystical word | Literal JSON reality |
|---|---|
| "Gemini decided to use a tool" | `candidates[0].content.parts[i].functionCall` key exists in the HTTP response body |
| "Gemini decided which tool" | The **string value** of `functionCall.name` — matched by the model against the `name` fields you put in `tools[0].functionDeclarations` |
| "Gemini figured out the arguments" | The **dict value** of `functionCall.args` — shaped to match the `parameters` JSON Schema you supplied, nothing more |
| "Gemini is running my code" | Never true. There is no code in the request JSON (Section 4) and no code execution anywhere on Google's infrastructure |
| "The agent executed the tool" | Your own `execute_tool()` Python function, a dict lookup + a function call, in your own process |
| "The agent gave the tool result back to the model" | A new HTTP request with a longer `contents` array containing a `{"functionResponse": {...}}` part |
| "Gemini decided on the final answer" | Same `generateContent` call, same `finishReason: "STOP"` field, just a `text` key instead of a `functionCall` key on the winning part |
| "You can force tool use" | `toolConfig.functionCallingConfig.mode` = the literal string `"ANY"`, `"NONE"`, or `"AUTO"` |
| "The agent loop" | A `while`/`for` loop **you** write, that keeps re-sending a growing `contents` array until a response arrives with no `functionCall` parts |

There is nothing else. Every framework you will ever use for Gemini tool calling — LangChain,
Google ADK, your own hand-rolled loop from Section 11 — reduces to exactly these JSON shapes,
sent to exactly this one REST method, some number of times in a row.

## 🎓 Day 4 Takeaway

1. Function calling is not a different API — it is `generateContent` given a `tools` field, and a
   response where a part contains `functionCall` instead of `text`.
2. Only the **interface** (`name`, `description`, `parameters` JSON Schema) ever leaves your
   machine. Never source code.
3. "Tool selection" and "final answer" are **the same underlying operation** — next-token
   generation over the `contents` array — producing two different, equally ordinary JSON shapes.
   `finishReason: "STOP"` looks identical either way.
4. The conversation has no server-side memory. Every round trip **resends the entire `contents`
   history**, including the model's own prior turns played back verbatim.
5. `toolConfig.functionCallingConfig.mode` (`AUTO` / `ANY` / `NONE`) is the one lever that overrides
   the model's free choice — and it's a plain string in the request JSON, nothing more.
6. Every "agent loop" you will ever use, in any framework, is Section 11's `while` loop wearing a
   different outfit.

### The one diagram to remember

```text
 YOUR MACHINE                                        GOOGLE'S SERVERS
 ─────────────                                        ─────────────────
 requests.post(generateContent,                       reads `contents` + `tools` as
   json={contents, tools})  ───────────────────────▶  plain context; predicts next tokens
                                                              │
 reads response JSON:                ◀───────────────────────┘
   part.functionCall  present?  ──▶ yes: run execute_tool() locally,
                                     append functionResponse, POST again
                                ──▶ no: part.text is the final answer, stop
```

## Official references

- Gemini API — Function calling guide: https://ai.google.dev/gemini-api/docs/function-calling
- Gemini API — `generateContent` REST reference: https://ai.google.dev/api/generate-content
- Gemini API — Tools overview: https://ai.google.dev/gemini-api/docs/tools
- Gemini API — `toolConfig` / `FunctionCallingConfig` reference: https://ai.google.dev/api/caching#FunctionCallingConfig

See also: [2-How_Gemini_Selects_And_Calls_Tools.ipynb](./2-How_Gemini_Selects_And_Calls_Tools.ipynb)
for the same concepts through the official `google-genai` SDK, and
[1-Building_AI_Agents.ipynb](./1-Building_AI_Agents.ipynb) for a full agent built on top of this
mechanism.